In [1]:
import json
from difflib import get_close_matches
from tabulate import tabulate

In [2]:
def load_data(file_path):
    """تحميل البيانات من ملف JSON."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"خطأ: لم يتم العثور على الملف في {file_path}")
        return {}
    except json.JSONDecodeError:
        print(f"خطأ: تعذر قراءة تنسيق JSON من {file_path}")
        return {}

In [3]:
def find_response(question, data):
    """البحث عن الاستجابة المناسبة من البيانات."""
    if not isinstance(data, list):
        return None
    questions = [entry["question"] for entry in data if "question" in entry]
    match = get_close_matches(question, questions, n=1, cutoff=0.6)
    if match:
        for entry in data:
            if entry.get("question") == match[0]:
                return entry.get("answer")
    return None

In [4]:
def detect_language(question, course_questions):
    """تحديد لغة السؤال حول المقررات (عربي أو إنجليزي)."""
    questions = [entry["question"] for entry in course_questions]
    match = get_close_matches(question, questions, n=1, cutoff=0.6)
    if match:
        for entry in course_questions:
            if entry["question"] == match[0]:
                return entry.get("language")
    return None

In [5]:
def show_courses(language, courses_data):
    """عرض قائمة المقررات بناءً على اللغة والمستوى الأكاديمي."""
    levels = [f"المستوى {i + 1}" if language == "ar" else f"Level {i + 1}" for i in range(8)]
    for i, level in enumerate(levels, start=1):
        courses = courses_data.get(f"level_{i}", courses_data.get(f"levle_{i}", []))
        if courses:
            print(f"{level}:")
            print(tabulate([[j + 1, course] for j, course in enumerate(courses)], headers=["No.", "Course Name"], tablefmt="fancy_grid"))
            print("\n")

In [6]:
def chatbot_interface():
    """واجهة المستخدم للشات بوت."""
    print("مرحبًا بك في المرشد الأكاديمي! يمكنك طرح استفساراتك حول الأداء الأكاديمي والمقررات.")

    greetings_data = load_data('/content/greetings.json')["greetings"]
    course_questions = load_data('/content/cores.json')["questions"]
    courses_data = load_data('/content/Words_For_Traning.json')

    while True:
        user_input = input("أنت: ").lower().strip()

        if user_input == "خروج":
            print("مع السلامة!")
            break

        # التحقق من التحية
        response = find_response(user_input, greetings_data)

        # التحقق من الأسئلة الأكاديمية إذا لم يتم العثور على تحية
        if not response:
            response = find_response(user_input, course_questions)

        # عرض المقررات إذا كان السؤال متعلقًا بالمقررات
        if not response:
            language = detect_language(user_input, course_questions)
            if language:
                email = input("من فضلك أدخل البريد الإلكتروني الخاص بك: ")
                while "@" not in email or "." not in email:
                    email = input("يرجى إدخال بريد إلكتروني صحيح: ")
                show_courses(language, courses_data)
                continue

        # رد افتراضي إذا لم يتم العثور على إجابة
        if not response:
            response = "عذرًا، لا أستطيع مساعدتك في هذا السؤال حاليًا."

        print("المرشد:", response)

In [8]:
if __name__ == "__main__":
    chatbot_interface()

مرحبًا بك في المرشد الأكاديمي! يمكنك طرح استفساراتك حول الأداء الأكاديمي والمقررات.
أنت: ازيك
المرشد: 🚀😊 بخير، شكرًا على السؤال! كيف أستطيع مساعدتك؟
أنت: عامل ايه
المرشد: عذرًا، لا أستطيع مساعدتك في هذا السؤال حاليًا.
أنت: السلام عليكم
المرشد: 🚀😊 وعليكم السلام! كيف يمكنني مساعدتك؟
أنت: اهلا 
المرشد: 🚀😊 مرحبًا بك! أنا هنا للمساعدة.
أنت: عمي و عم البلد
المرشد: 🚀😊 حبيب قلبي ، ! كيف أستطيع مساعدتك؟
أنت: hi
المرشد: Hi there! How can I help you 🚀😊?
أنت: how are you 
المرشد: I'm fine, thank you! How can I assist you? 🚀😊
أنت: ما هي المقررات
من فضلك أدخل البريد الإلكتروني الخاص بك: sayed
يرجى إدخال بريد إلكتروني صحيح: sayed@.gamil
المستوى 1:
╒═══════╤════════════════════════════════════╕
│   No. │ Course Name                        │
╞═══════╪════════════════════════════════════╡
│     1 │ Differentiation and Integration    │
├───────┼────────────────────────────────────┤
│     2 │ Properties of matter and Heat      │
├───────┼────────────────────────────────────┤
│     3 │ Statics             